# Exploratory Data Analysis (EDA)
## AI-Powered Phishing Email Detector

This notebook explores the cleaned dataset and analyzes patterns in phishing vs. legitimate emails.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import re

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
data_dir = Path('../data')
df = pd.read_csv(data_dir / 'cleaned_emails.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Class distribution
class_counts = df['label'].value_counts()
print(f"Class Distribution:")
print(class_counts)
print(f"\nPhishing Ratio: {(class_counts.get(1, 0) / len(df) * 100):.2f}%")
print(f"Legitimate Ratio: {(class_counts.get(0, 0) / len(df) * 100):.2f}%")

# Plot class distribution
fig, ax = plt.subplots(figsize=(8, 5))
labels_map = {0: 'Legitimate', 1: 'Phishing'}
class_counts.rename(labels_map).plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Class Distribution (Legitimate vs. Phishing)', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Email Type')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# Plot as pie chart
fig, ax = plt.subplots(figsize=(8, 5))
class_counts.rename(labels_map).plot(kind='pie', ax=ax, labels=['Legitimate', 'Phishing'], 
                                       colors=['green', 'red'], autopct='%1.1f%%')
ax.set_title('Dataset Class Balance', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Helper function to extract words from text
def get_words(text):
    """Extract words from email text."""
    if pd.isna(text):
        return []
    # Remove URLs, emails, numbers, and special characters
    text = re.sub(r'http\S+|www\S+|\S+@\S+|\d+|[^a-zA-Z\s]', '', str(text))
    return text.lower().split()

# Extract words for each class
phishing_words = []
legitimate_words = []

for idx, row in df.iterrows():
    words = get_words(row['cleaned_text'])
    if row['label'] == 1:
        phishing_words.extend(words)
    else:
        legitimate_words.extend(words)

# Get top 20 words for each class
phishing_counter = Counter(phishing_words)
legitimate_counter = Counter(legitimate_words)

top_phishing = phishing_counter.most_common(20)
top_legitimate = legitimate_counter.most_common(20)

print(f"Top 20 words in PHISHING emails:")
for word, count in top_phishing:
    print(f"  {word}: {count}")

print(f"\nTop 20 words in LEGITIMATE emails:")
for word, count in top_legitimate:
    print(f"  {word}: {count}")

In [ ]:
# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Phishing words
phishing_words_df = pd.DataFrame(top_phishing, columns=['word', 'count']).sort_values('count')
axes[0].barh(phishing_words_df['word'], phishing_words_df['count'], color='red')
axes[0].set_title('Top 20 Most Frequent Words in PHISHING Emails', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Frequency')

# Legitimate words
legitimate_words_df = pd.DataFrame(top_legitimate, columns=['word', 'count']).sort_values('count')
axes[1].barh(legitimate_words_df['word'], legitimate_words_df['count'], color='green')
axes[1].set_title('Top 20 Most Frequent Words in LEGITIMATE Emails', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze email length and other features
df['text_length'] = df['cleaned_text'].str.len()
df['word_count'] = df['cleaned_text'].str.split().str.len()
df['url_count'] = df['cleaned_text'].str.count(r'http\S+|www\S+')
df['email_count'] = df['cleaned_text'].str.count(r'\S+@\S+')
df['has_urgency'] = df['cleaned_text'].str.contains('urgent|immediate|click|verify|confirm|act now', case=False, na=False).astype(int)

print(f"\nFeature Statistics by Class:")
print(df.groupby('label')[['text_length', 'word_count', 'url_count', 'email_count', 'has_urgency']].describe())

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features = ['text_length', 'word_count', 'url_count', 'email_count']

for idx, feature in enumerate(features):
    ax = axes[idx // 3, idx % 3]
    
    phishing_data = df[df['label'] == 1][feature]
    legitimate_data = df[df['label'] == 0][feature]
    
    ax.hist([legitimate_data, phishing_data], bins=30, label=['Legitimate', 'Phishing'], 
            color=['green', 'red'], alpha=0.7)
    ax.set_title(f'{feature} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Urgency flag distribution
ax = axes[1, 2]
urgency_data = df.groupby(['label', 'has_urgency']).size().unstack(fill_value=0)
urgency_data.T.plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Urgency Language Distribution', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xticklabels(['No Urgency', 'Has Urgency'], rotation=0)
ax.legend(['Legitimate', 'Phishing'])

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, feature in enumerate(['text_length', 'word_count', 'url_count', 'email_count']):
    ax = axes[idx // 2, idx % 2]
    
    data_to_plot = [df[df['label'] == 0][feature].dropna(), 
                     df[df['label'] == 1][feature].dropna()]
    
    bp = ax.boxplot(data_to_plot, labels=['Legitimate', 'Phishing'], patch_artist=True)
    
    # Color the boxes
    colors = ['green', 'red']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(f'{feature} Distribution (Box Plot)', fontsize=12, fontweight='bold')
    ax.set_ylabel(feature)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print(f"\n{'='*60}")
print(f"SUMMARY STATISTICS")
print(f"{'='*60}")
print(f"\nTotal emails: {len(df):,}")
print(f"Legitimate emails: {len(df[df['label'] == 0]):,} ({(len(df[df['label'] == 0])/len(df)*100):.2f}%)")
print(f"Phishing emails: {len(df[df['label'] == 1]):,} ({(len(df[df['label'] == 1])/len(df)*100):.2f}%)")
print(f"\nClass balance ratio: {len(df[df['label'] == 0])/len(df[df['label'] == 1]):.2f}:1 (Legitimate:Phishing)")
print(f"\nAverage text length:")
print(f"  Legitimate: {df[df['label'] == 0]['text_length'].mean():.0f} chars")
print(f"  Phishing: {df[df['label'] == 1]['text_length'].mean():.0f} chars")
print(f"\nEmails with URLs:")
print(f"  Legitimate: {(df[df['label'] == 0]['url_count'] > 0).sum()} ({(df[df['label'] == 0]['url_count'] > 0).mean()*100:.1f}%)")
print(f"  Phishing: {(df[df['label'] == 1]['url_count'] > 0).sum()} ({(df[df['label'] == 1]['url_count'] > 0).mean()*100:.1f}%)")
print(f"\nEmails with urgency language:")
print(f"  Legitimate: {df[df['label'] == 0]['has_urgency'].sum()} ({df[df['label'] == 0]['has_urgency'].mean()*100:.1f}%)")
print(f"  Phishing: {df[df['label'] == 1]['has_urgency'].sum()} ({df[df['label'] == 1]['has_urgency'].mean()*100:.1f}%)")

## Key Findings

- **Dataset Balance**: The dataset is reasonably balanced between legitimate and phishing emails
- **Text Length**: Phishing emails tend to be shorter than legitimate emails
- **URLs**: Phishing emails contain significantly more URLs than legitimate emails
- **Urgency Language**: Phishing emails use urgent language much more frequently
- **Word Patterns**: Distinctive vocabulary patterns appear in both classes

These features will be used for training the machine learning models.